# 06 — Text Classification with TF-IDF

Before jumping straight to an LLM, a strong baseline matters. TF-IDF + logistic regression is fast, interpretable and often surprisingly competitive for small labelled text datasets.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

rows = [
    ('payment failed and I was charged twice', 'billing'),
    ('please refund the duplicate card charge', 'billing'),
    ('my invoice total is incorrect', 'billing'),
    ('why did my subscription renew', 'billing'),
    ('the app crashes when I open settings', 'technical'),
    ('login page keeps showing an error', 'technical'),
    ('the upload button is not working', 'technical'),
    ('I cannot reset my password', 'technical'),
    ('how do I change my delivery address', 'account'),
    ('I need to update my email address', 'account'),
    ('please close my account', 'account'),
    ('where can I change notification settings', 'account'),
] * 8
df = pd.DataFrame(rows, columns=['text', 'label'])
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.25, stratify=df['label'], random_state=42
)
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )),
    ('classifier', LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

examples = [
    'I have been billed two times this month',
    'the software freezes after I log in',
    'I want to update the email on my profile',
]
for text, label in zip(examples, model.predict(examples)):
    print(f'{label:10s} <- {text}')

## What TF-IDF is doing

`TfidfVectorizer` converts documents into sparse numeric vectors. A word receives more weight when it is common in one document but not common across every document. Adding bigrams lets the model learn short phrases such as `card charge` or `reset password`.

For a real dataset I would also inspect class imbalance, duplicates/near-duplicates, train/test contamination, ambiguous labels and drift. I would compare this baseline with a transformer only if the extra complexity is justified.